In [0]:
df_data = spark.sql("FROM supply_chain.1_bronze.raw_supply_chain")

df_data.display()

In [0]:
print(df_data.columns)

In [0]:
df_data.select(
    "Customer Email",
    "Customer Country",
    "Benefit per order",
    "shipping date (DateOrders)",
).limit(10).display()

In [0]:
df_data.select("Product Description").distinct().display()

In [0]:
df_metadata = spark.sql("FROM supply_chain.1_bronze.metadata")

df_metadata.display()

In [0]:
df_logs = spark.sql("FROM supply_chain.1_bronze.raw_access_logs")

df_logs.display()

## Summary of cleaning steps

We note down what to be cleaned based on looking at the data, reading metadata and doing some EDA. This list is not comprehensive, I don't want to take away all the fun for you :)

I will leave it to you to continue clean this data

**Drop**
- Customer Email (data is masked)
- Customer Password
- Product Description (all are nulls) 

**Rounding errors**
- Order Item Product Price -> 2 decimals
- Order Item Product Price -> 2 decimals
- Benefit per order -> 2 decimals
- Sales per customer -> 2 decimals 
...

**Customer country** 
- EE. UU. -> United States as it is the spanish abbreviation

**Nulls**
- fill Customer Lname with missing
- fill Customer Zipcode with missing
- fill Order Zipcode with missing

**Data type**
- shipping date - string -> Timestamp

**Derived columns** - will not remove them in silver layer, but will compute them in Gold layer instead
- Order Item Total - derived from (Order Item Quantity)*(Order Item Product Price)*(1-Order Item Discount Rate), calculate and test this 
...

**Rename columns**
- use snake_case convention instead (this is a design choice)

In [0]:
df_data.select(
    "Customer Email",
    "Customer Password",
    "Product Description",
    "Order Item Product Price"
).limit(10).display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df_data.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df_data.columns]
)

null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

#### To snake_case

In [0]:
import re 

def to_snake_case(name):
    name = name.strip().casefold()
    name = re.sub(r"[()]+", "", name)
    name = re.sub(r"[\s]+", "_", name)
    return name

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

to_snake_case(" Customer    Email  AcCount ")

In [0]:
df_cleaned_columns = rename_columns_to_snake_case(df_data)
df_cleaned_columns.display()

In [0]:
from pyspark.sql.functions import to_timestamp, col, coalesce, lit, when, round

# 6/19/2017 4:41

df_cleaned = (
    df_cleaned_columns.withColumn(
        "shipping_date", to_timestamp("shipping_date_dateorders", "M/d/yyyy H:mm")
    )
    .withColumn(
        "order_zipcode", coalesce(col("order_zipcode").cast("string"), lit("unknown"))
    )
    .withColumn(
        "customer_zipcode",
        coalesce(col("customer_zipcode").cast("string"), lit("unknown")),
    ).withColumn(
        "customer_country", when(col("customer_country") == "EE. UU.", "United States").otherwise(col("customer_country"))
    ).withColumn(
        "order_date", to_timestamp("order_date_dateorders", "M/d/yyyy H:mm")
    ).withColumn(
        "benefit_per_order_dec", round("benefit_per_order", 2)
    ).withColumn(
        "sales_per_customer_dec", round("sales_per_customer", 2)
    ).withColumn(
        "order_item_product_price_dec", round("order_item_product_price", 2)
    ).drop(
        "customer_email",
        "customer_password",
        "product_description",
        "shipping_date_dateorders",
        "order_date_dateorders",
        "benefit_per_order",
        "sales_per_customer",
        "order_item_product_price"
    )
)

df_cleaned.display()